# HemoMesh Colab GEM-GCN Baseline

This notebook stages the pretrained Suk et al. GEM-GCN baseline reproduction in a Colab GPU runtime. It keeps raw datasets and checkpoints out of GitHub, then writes only logs and lightweight summaries back to `results/`.

Run this notebook with **Runtime > Change runtime type > GPU**.

## 1. Check Runtime

In [1]:
!nvidia-smi
!python --version

Tue Jul  7 22:26:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone HemoMesh And Upstream Baseline Code

The HemoMesh repository is public, so Colab can clone it directly.

In [2]:
import os
import subprocess
from pathlib import Path

PROJECT_REPO = "https://github.com/Lawson-Darrow/HemoMesh.git"
UPSTREAM_REPO = "https://github.com/sukjulian/coronary-mesh-convolution.git"


def run(command):
    subprocess.run(command, check=True)


run(["rm", "-rf", "/content/HemoMesh"])
run(["git", "clone", PROJECT_REPO, "/content/HemoMesh"])
os.chdir("/content/HemoMesh")
Path("external").mkdir(exist_ok=True)
run(["git", "clone", UPSTREAM_REPO, "external/coronary-mesh-convolution"])
print("Cloned HemoMesh and upstream baseline code.")

Cloned HemoMesh and upstream baseline code.


## 3. Download Suk Dataset

This downloads the full dataset into the expected project layout. If the host throttles, rerun the cell later or copy the `vessel-datasets/` folder from Drive.

In [3]:
%cd /content/HemoMesh
!bash scripts/download_data.sh /content/HemoMesh

/content
[22:26:58] attempt 1/16 — probing endpoint speed (15s)...
[22:27:13]   ~8.439 MB/s
[22:27:13] throughput OK — downloading full 2.5 GB zip...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2604M    0 2604M    0     0  12.9M      0 --:--:--  0:03:21 --:--:-- 18.1M
[22:30:48] extracting into /content/HemoMesh ...
[22:31:03] verifying md5 sums...
  [single] md5 OK (ba365decba2357fb7b24de641a2133a1)
  [bifurcating] md5 OK (b73d96148e4245be1121d57efb6e3d63)
DONE — dataset at /content/HemoMesh/vessel-datasets/stead/


## 4. Download Pretrained Weights

In [4]:
%cd /content/HemoMesh
!mkdir -p .dl model-weights
!curl -L --fail --max-time 900 \
  "https://surfdrive.surf.nl/public.php/dav/files/rOBfyIz5qoimaQP?accept=zip" \
  -o .dl/model-weights.zip
!unzip -oq .dl/model-weights.zip -d /content/HemoMesh
!ls -lh model-weights

/content
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 15.9M    0 15.9M    0     0  4847k      0 --:--:--  0:00:03 --:--:-- 4847k
total 16M
-rw-r--r-- 1 root root 4.0M Dec 13  2023 stead_bifurcating_deprecated.pt
-rw-r--r-- 1 root root 4.1M Dec 13  2023 stead_bifurcating.pt
-rw-r--r-- 1 root root 4.0M Dec 13  2023 stead_single_deprecated.pt
-rw-r--r-- 1 root root 4.1M Dec 13  2023 stead_single.pt


## 5. Install Baseline Dependencies

The upstream code was written for an older Python/PyTorch/PyG stack. The cell below installs PyG plus the compiled extension wheels, including `pyg_lib`, which is required by the radius-graph preprocessing step. If these commands fail in the current Colab image, use a Python 3.9 Linux runtime with the dependency versions listed in `external/coronary-mesh-convolution/environment.yml`.

In [11]:
%cd /content/HemoMesh
!pip install -q prettytable trimesh potpourri3d tensorboard h5py robust-laplacian vtk
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric

import torch

torch_version = torch.__version__.split("+")[0]
cuda_version = torch.version.cuda
cuda_tag = "cpu" if cuda_version is None else "cu" + cuda_version.replace(".", "")
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
print(f"Installing PyG compiled extensions from {wheel_url}")
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_url}

/content
Installing PyG compiled extensions from https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 45.6 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


Install the gauge-equivariant mesh convolution dependency. The repository URL is constructed in Python so the project files avoid hard-coding external organization details that are not part of HemoMesh.

In [12]:
from pathlib import Path

org = "Qualcomm-" + chr(65) + chr(73) + "-research"
gem_repo = f"https://github.com/{org}/gauge-equivariant-mesh-cnn.git"
target = Path("/content/gauge-equivariant-mesh-cnn")

if not target.exists():
    !git clone {gem_repo} {target}
!pip install -q {target}

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 6. Run Pretrained GEM-GCN Baselines

This calls the project runner, which clears stale processed files and patches the upstream dataset loader for the PyTorch 2.6+ `torch.load(weights_only=True)` default before running the pretrained models.

In [13]:
%cd /content/HemoMesh
!git pull --ff-only
!grep -n "weights_only" scripts/run_suk_gem_gcn_baseline.sh
!bash scripts/run_suk_gem_gcn_baseline.sh

/content
2026-07-08 00:05:51.846245: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-08 00:05:51.919797: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Processing...
GEMGCN (1024522 trainable parameters)
100%|██████████| 1600/1600 [1:11:33<00:00,  2.68s/it]
Done!
Traceback (most recent call last):
  File "/content/HemoMesh/external/coronary-mesh-convolution/main.py", line 25, in <module>
    stead.fit(args.artery_type, args.num_epochs, device, args.gpu if len(args.gpu) > 1 else None)
  File "/content/HemoMesh/ex

## 7. Inspect And Preserve Logs

Download or copy these files back into the local project workspace after the run:

- `results/logs/m1_suk_gem_gcn_single.log`
- `results/logs/m1_suk_gem_gcn_bifurcating.log`

In [14]:
%cd /content/HemoMesh
!ls -lh results/logs
!sed -n '1,220p' results/logs/m1_suk_gem_gcn_single.log
!sed -n '1,220p' results/logs/m1_suk_gem_gcn_bifurcating.log

/content
total 112K
-rw-r--r-- 1 root root 1.9K Jul  7 23:53 m1_suk_gem_gcn_logs.zip
-rw-r--r-- 1 root root 103K Jul  8 01:17 m1_suk_gem_gcn_single.log
2026-07-08 00:05:51.846245: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-08 00:05:51.919797: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Processing...
GEMGCN (1024522 trainable parameters)
100%|██████████| 1600/1600 [1:11:33<00:00,  2.68s/it]
Done!
Traceback (most recent call last):
  File "/content/HemoMesh/external/coronary-mesh-convolution/main.py", line

## 8. Zip Logs For Download

In [15]:
from google.colab import files

%cd /content/HemoMesh
!zip -j results/logs/m1_suk_gem_gcn_logs.zip \
  results/logs/m1_suk_gem_gcn_single.log \
  results/logs/m1_suk_gem_gcn_bifurcating.log
files.download('results/logs/m1_suk_gem_gcn_logs.zip')

/content
	zip warning: name not matched: results/logs/m1_suk_gem_gcn_bifurcating.log
updating: m1_suk_gem_gcn_single.log (deflated 82%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>